In [2]:
import json
import os
from agents import Agent, Runner, handoff
from dotenv import load_dotenv
import requests
import asyncio
import openai
from agents import function_tool
from pydantic import BaseModel
import random
from agents import trace
from agents import RunContextWrapper
import time


load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")


if not api_key:
    raise ValueError(
        "OpenAI API key not found. Please set the OPENAI_API_KEY environment variable."
    )

In [2]:
@function_tool
def get_weather(city: str) -> str:
    """Get the Current Weather for a city"""

    api_key = os.getenv("OPEN_WEATHER_API")

    response = requests.get(
        f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={api_key}&units=metric"
    ).json()

    weather = response.get("weather")[0]["description"]
    temp = response["main"]["temp"]
    return f"The current weather in {city} is {weather} with a temperature of {temp}°C."

In [3]:
openai.api_key = api_key

agent = Agent(
    name="Psychology Guide",
    instructions="You are a psychology expert. Provide the most concise explanations and insights on psychological concepts.",
    model="gpt-5-mini",
)


result = await Runner.run(agent, "What is Eisenhower Matrix?")
print(result.final_output)

The Eisenhower Matrix (aka Urgent‑Important Matrix) is a simple prioritization tool attributed to U.S. President Dwight D. Eisenhower and popularized by Stephen Covey. It helps decide what to do, schedule, delegate, or delete by sorting tasks along two axes: urgency and importance.

Quadrants
- Important + Urgent — Do now (crises, deadlines).
- Important + Not urgent — Schedule (planning, long‑term goals).
- Not important + Urgent — Delegate (interruptions, some requests).
- Not important + Not urgent — Eliminate (time‑wasters, low‑value activities).

Quick examples
- Do now: project due today, medical emergency.
- Schedule: career development, exercise, relationship time.
- Delegate: routine admin, meeting prep someone else can do.
- Eliminate: scrolling social media, unnecessary meetings.

Why it helps (psychological rationale)
- Reduces decision fatigue by giving a clear rule for action.
- Shifts focus from reactive (urgent) to proactive (important), improving long‑term goal pursuit

In [13]:
class cars(BaseModel):
    make: str
    generations: str
    year_released: int
    type: str
    color: str
    still_produced: bool


car_agent = Agent(
    name="Car Agent",
    instructions="You are a car expert. You'll be given a car name and you job is to provide basic information about that car.",
    model="gpt-4o-mini",
    output_type=cars,
)


result1 = await Runner.run(
    car_agent,
    "tell me about the BMW M5 Competition?",
)
print("Structured Output :", result1.final_output, "\n")

data = result1.final_output.model_dump()
print("Python Object : ", data, "\n")

data1 = json.dumps(data, indent=2)
print("json Schema :", data1, "\n")
# schema = result1.final_output.model_json_schema()

Structured Output : make='BMW' generations='F90' year_released=2018 type='Sedan' color='Various (including Black, Silver, Blue, White)' still_produced=True 

Python Object :  {'make': 'BMW', 'generations': 'F90', 'year_released': 2018, 'type': 'Sedan', 'color': 'Various (including Black, Silver, Blue, White)', 'still_produced': True} 

json Schema : {
  "make": "BMW",
  "generations": "F90",
  "year_released": 2018,
  "type": "Sedan",
  "color": "Various (including Black, Silver, Blue, White)",
  "still_produced": true
} 



In [14]:
payload = json.dumps({"data": data}, indent=2)

In [15]:
print(
    f"This is python Object {data} \n and this is json formatted string of same data : {payload}"
)

This is python Object {'make': 'BMW', 'generations': 'F90', 'year_released': 2018, 'type': 'Sedan', 'color': 'Various (including Black, Silver, Blue, White)', 'still_produced': True} 
 and this is json formatted string of same data : {
  "data": {
    "make": "BMW",
    "generations": "F90",
    "year_released": 2018,
    "type": "Sedan",
    "color": "Various (including Black, Silver, Blue, White)",
    "still_produced": true
  }
}


In [18]:
carInfo_agent = Agent(
    name="Car Info Agent",
    instructions="You are a car information expert. You'll be given a Schema about a car and your job is to provide brief information about similar cars that also have those qualities.",
    model="gpt-4o-mini",
)

result = await Runner.run(carInfo_agent, payload)
print(result.final_output)

Here are some similar cars that share qualities with the 2018 BMW F90 generation 5 Series sedan:

1. **Audi A6 (C8)**
   - **Year Released:** 2019
   - **Type:** Sedan
   - **Colors:** Various (including Black, Silver, Blue, White)
   - **Still Produced:** Yes
   - **Highlights:** Known for its luxury, advanced technology, and Quattro all-wheel drive system.

2. **Mercedes-Benz E-Class (W213)**
   - **Year Released:** 2017
   - **Type:** Sedan
   - **Colors:** Various (including Black, Silver, Blue, White)
   - **Still Produced:** Yes
   - **Highlights:** Offers a balance of performance and comfort with a sophisticated interior.

3. **Lexus ES ( seventh generation)**
   - **Year Released:** 2018
   - **Type:** Sedan
   - **Colors:** Various (including Black, Silver, Blue, White)
   - **Still Produced:** Yes
   - **Highlights:** Renowned for its reliability, quiet ride, and luxurious feel.

4. **Genesis G80 (second generation)**
   - **Year Released:** 2017
   - **Type:** Sedan
   - **C

## Using Web Search tool

In [19]:
from agents import WebSearchTool

NewsSearchAgent = Agent(
    name="News Search Agent",
    instructions="You are a news search agent. You'll be given a query and your job is to find the latest news articles related to that query.",
    model="gpt-4o-mini",
    tools=[WebSearchTool()],
)


while True:
    query = input("Enter your Query (or exit using 'quit')")
    if query.lower() == "quit":
        break

    result = await Runner.run(NewsSearchAgent, query)

    print(result.final_output)

## Using Handoffs

In [6]:
import asyncio
from agents import RunContextWrapper


class Quote(BaseModel):
    personality_name: str
    quote_attributed: str
    language: str


translator_agent_arabic = Agent(
    name="Arabic Translator Agent",
    handoff_description="Translates the given text in Arabic",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into Arabic language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)


translator_agent_urdu = Agent(
    name="Urdu Translator Agent",
    handoff_description="Translates the given text in Urdu",
    instructions="""You are a translator agent. You'll be given a quote as {"Name": "<the quote of him>"} and your job is to translate the whole text into Urdu language. and give the final output as {"Name": "<the translated quote>"}.""",
    model="gpt-4o-mini",
    output_type=Quote,
)


def on_urdu_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to Urdu translator agent")


def on_arabic_handoff(ctx: RunContextWrapper[None]):
    print("Handing off to Arabic translator agent")


quote_agent = Agent(
    name="Quote Agent",
    instructions="""
    You are a quote agent. You'll be given a person name and a language name in which it is required in. 
        Find one relevant quote attributed to that name. and hand it to the required translator agent
        if the language is 'Arabic', always hand off to the Arabic translator agent.
        If the language is 'Urdu', always hand off to the Urdu translator agent.
    """,
    model="gpt-5-mini",
    output_type=Quote,
    handoffs=[
        handoff(
            agent=translator_agent_arabic,
            on_handoff=(on_arabic_handoff),
        ),
        handoff(
            agent=translator_agent_urdu,
            on_handoff=(on_urdu_handoff),
        ),
    ],
)
data = []
while True:
    person = input("Enter a personality name to get a quote(or exit using 'quit') ")

    if person.lower() == "quit":
        break
    language = (
        input("Enter the language (arabic/urdu) for translation: ").strip().lower()
    )
    if language.lower() == "quit":
        break

    quote = await Runner.run(
        quote_agent, "Give me " + person + "'s quote in " + language + " language"
    )
    print(quote.final_output)
    data.append(quote.final_output.model_dump())

personality_name='Umar ibn al-Khattab (R.A)' quote_attributed='No amount of guilt can change the past and no amount of anxiety can change the future.' language='English'


## Using Handoffs Again

In [6]:
class ManagerEscalation(BaseModel):
    issue: str  # the issue being escalated
    reason: str  # the reason for escalation


@function_tool
def create_ticket(issue: str):
    """Create a ticket for the escalated issue"""
    # Simulate ticket creation
    print(f"Ticket created for issue: {issue}")
    return f"Ticket created successfully ID {random.randint(10000, 99999)}"


manager_agent = Agent(
    name="Manager Agent",
    handoff_description="Handles Escalated Issues that requires managerial oversight",
    instructions="""
    you handle escalated customer issues that the initial customer service agent could not resolve.
    you will recieve the issue and the reason for escalation if the issue could not be resolved for the 
    customer create a ticket and inform the customer.
    """,
    tools=[create_ticket],
    model="gpt-4o-mini",
)


def on_manager_handoff(ctx: RunContextWrapper[None], input: ManagerEscalation):
    print("Escalating to the Manager Agent : \t", input.issue)
    print("Reason For Escalation : \t", input.reason)


customer_service_agent = Agent(
    name="Customer Service Agent",
    handoff_description="Handles initial customer inquiries and issues",
    instructions="""
    you are the first point of contact for customer inquiries.
    if you cannot resolve the issue, escalate it to the manager agent.
    """,
    model="gpt-4o-mini",
    handoffs=[
        handoff(
            manager_agent,
            input_type=ManagerEscalation,
            on_handoff=on_manager_handoff,
        )
    ],
)

with trace("Customer Service Workflow"):
    result = await Runner.run(
        customer_service_agent, "I need a refund but the website is blank"
    )
    print(result.final_output)

Escalating to the Manager Agent : 	 Customer is unable to access the website to initiate a refund.
Reason For Escalation : 	 Website issue affecting customer service.
Ticket created for issue: Customer requests a refund, but the website is currently blank and inaccessible.
I've created a ticket for your issue with ID **85487** regarding your request for a refund. Since the website is currently blank, our team will look into it and get back to you as soon as possible. Thank you for your patience!


## Tracing

In [ ]:
basic_agent = Agent(
    name="Basic Agent",
    instructions="""
    you are a helpful assitant that responds in the most concise way possible to address user inquiries.
    """,
    model="gpt-4o-mini",
)
with trace("Basic Agent Workflow"):
    result = await Runner.run(
        basic_agent, "who is the owner of the club manchester city and manchester united?"
)

print(result.final_output)

Manchester City is owned by the City Football Group, led by Sheikh Mansour. Manchester United is primarily owned by the Glazer family.


## Streaming

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent

openai.api_key = api_key

agent = Agent(
    name="Psychology Guide",
    instructions="You are a psychology expert. Provide the most concise explanations and insights on psychological concepts.",
    model="gpt-4o-mini",
)


result = Runner.run_streamed(agent, "Give me 5 jokes on dark Psychology")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Sure! Here are five light-hearted jokes inspired by dark psychology themes:

1. Why did the manipulator bring a ladder to therapy?
   Because they wanted to take their issues to new heights!

2. What did the narcissist say to their reflection?
   "You're the best company I've ever had!"

3. Why don’t psychopaths play hide and seek?
   Because good luck hiding when they know all your secrets!

4. How does a gaslighter propose?
   "I thought you loved surprises, but clearly, you love being wrong!"

5. Why did the sadist become a psychologist?
   They wanted to help others feel the pain they never let go of! 

Remember, humor can walk a fine line, so it’s always good to gauge the audience!

In [ ]:
from openai.types.responses import ResponseTextDeltaEvent
from agents import ItemHelpers

openai.api_key = api_key

agent = Agent(
    name="Psychology Guide",
    instructions="You are a psychology expert. Provide the most concise explanations and insights on psychological concepts.",
    model="gpt-4o-mini",
)


result = Runner.run_streamed(agent, "Give me 5 jokes on dark Psychology")

print("____Streaming____")

async for event in result.stream_events():
    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print(f"Agent Updated : {event.new_agent.name}")
        continue
    
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print("-Tool Was Called")
        if event.item.type == "tool_call_output_item":
            print(f"Tool Output {event.item.output}")
        if event.item.type == "message_output_item":
            print(f"Message Ouptput : \n {ItemHelpers.text_message_output(event.item)}")
        else:
            pass

print("____Streaming Stop____")

____Streaming____
Agent Updated Psychology Guide
Message Ouptput : 
 Sure! Here are five dark humor jokes related to psychology:

1. Why don’t sociopaths ever get lost?
   Because they always have a *twisted* sense of direction.

2. Why did the narcissist bring a ladder to therapy?
   To elevate their self-esteem... literally!

3. How do you know you’re dating a psychologist?
   They’ll diagnose you even while sharing your dessert.

4. What did the therapist say to the murderer?
   “You really need to work on your *issues*… and I mean *issues*!”

5. Why did the paranoid person bring a map to the therapist’s office?
   To track their *delusions*! 

(Note: Remember that humor involving sensitive topics should always be approached with caution and respect.)
____Streaming Stop____


In [7]:
from agents import ItemHelpers

In [13]:
class ManagerEscalation(BaseModel):
    issue: str  # the issue being escalated
    reason: str  # the reason for escalation


@function_tool
def create_ticket(issue: str):
    """Create a ticket for the escalated issue"""
    # Simulate ticket creation
    print(f"Ticket created for issue: {issue}")
    return f"Ticket created successfully ID {random.randint(10000, 99999)}"


manager_agent = Agent(
    name="Manager Agent",
    handoff_description="Handles Escalated Issues that requires managerial oversight",
    instructions="""
    you handle escalated customer issues that the initial customer service agent could not resolve.
    you will recieve the issue and the reason for escalation if the issue could not be resolved for the 
    customer create a ticket and inform the customer.
    """,
    tools=[create_ticket],
    model="gpt-4o-mini",
)


def on_manager_handoff(ctx: RunContextWrapper[None], input: ManagerEscalation):
    print("Escalating to the Manager Agent : \t", input.issue)
    print("Reason For Escalation : \t", input.reason)


customer_service_agent = Agent(
    name="Customer Service Agent",
    handoff_description="Handles initial customer inquiries and issues",
    instructions="""
    you are the first point of contact for customer inquiries.
    if you cannot resolve the issue, escalate it to the manager agent.
    """,
    model="gpt-4o-mini",
    handoffs=[
        handoff(
            manager_agent,
            input_type=ManagerEscalation,
            on_handoff=on_manager_handoff,
        )
    ],
)

result = Runner.run_streamed(
    customer_service_agent, "I need a refund but the website is blank"
)

print("___Streaming_ON____\n")

async for event in result.stream_events():
    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print(f"Agent Updated {event.new_agent.name} \n")
        continue
    elif event.type == "run_item_stream_event":
        item = event.item
        if item.type == "tool_call_item":
            print(f"Tool Called  by : {item.agent.name} \n")
        if item.type == "tool_call_output_item":
            print(f"\n -Tool Ouptut : {item.output}\n")
        if item.type == "handoff_call_item":
            print(f"handoff initiating towards : {item.agent.name}\n")
        if item.type == "handoff_output_item":
            time.sleep(0.5)
            print(f"Handoff to  {item.agent.name} \n")
        if item.type == "message_output_item":
            print(f"-- Message Output : {ItemHelpers.text_message_output(item)}\n")
        else:
            pass
print("____Streaming_OFF____")

___Streaming_ON____

Agent Updated Customer Service Agent 

handoff initiating towards : Customer Service Agent

Escalating to the Manager Agent : 	 Customer needs a refund but the website is not functioning properly (blank page).
Reason For Escalation : 	 Technical issue preventing customer from processing a refund.
Handoff to  Customer Service Agent 

Agent Updated Manager Agent 

Tool Called  by : Manager Agent 

Ticket created for issue: Customer needs a refund but the website is blank, preventing the refund process from being completed.

 -Tool Ouptut : Ticket created successfully ID 28784

-- Message Output : I have created a ticket for your issue regarding the refund (Ticket ID: 28784). It looks like there is a technical issue with the website showing a blank page. Our team will look into it and get back to you as soon as possible. Thank you for your patience!

____Streaming_OFF____
